# 03 · LangGraph — agente con auto-corrección de SQL

La chain del notebook 01 tenía un problema: si el LLM genera SQL inválido, **falla y punto**. En producción eso no sirve: los LLMs se equivocan, y mucho.

**LangGraph** permite construir un **grafo de estados** donde:

- Cada nodo es una función que actualiza el estado.
- Las aristas pueden ser **condicionales**: según el estado, el grafo va a un nodo u otro.
- Puedes crear **loops** con límite de iteraciones.

En este notebook construimos un agente que **reintenta** cuando el SQL falla, alimentando el error al LLM para que corrija.

In [1]:
import sys
import os
import tempfile
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from sqlalchemy import create_engine, text
db_path = Path(tempfile.gettempdir()) / "demo_chat_bd.db"
os.environ["DATABASE_URL"] = f"sqlite:///{db_path}"
engine = create_engine(os.environ["DATABASE_URL"], future=True)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS productos"))
    conn.execute(text("CREATE TABLE productos (id INTEGER PRIMARY KEY, nombre TEXT, precio REAL)"))
    for i, (n, p) in enumerate([("a", 10), ("b", 20), ("c", 30)], start=1):
        conn.execute(text("INSERT INTO productos VALUES (:i, :n, :p)"), {"i": i, "n": n, "p": p})

from chat_bd import config, sql_tools
config.DATABASE_URL = os.environ["DATABASE_URL"]
sql_tools.DATABASE_URL = os.environ["DATABASE_URL"]
sql_tools.crear_engine.cache_clear()
print("BD lista")

BD lista


## 1 · El estado compartido

LangGraph requiere que declares la "forma" del estado que viaja entre nodos. Usamos un `TypedDict`:

In [2]:
from chat_bd.agent_graph import AgenteState

# AgenteState define: pregunta, schema, sql, resultado, respuesta, error, intentos, max_intentos.
print(AgenteState.__annotations__)

{'pregunta': ForwardRef('str', module='chat_bd.agent_graph'), 'schema': ForwardRef('str', module='chat_bd.agent_graph'), 'sql': ForwardRef('str', module='chat_bd.agent_graph'), 'resultado': ForwardRef('pd.DataFrame', module='chat_bd.agent_graph'), 'respuesta': ForwardRef('str', module='chat_bd.agent_graph'), 'error': ForwardRef('str | None', module='chat_bd.agent_graph'), 'intentos': ForwardRef('int', module='chat_bd.agent_graph'), 'max_intentos': ForwardRef('int', module='chat_bd.agent_graph')}


## 2 · El grafo

```
START
  ↓
generar_sql
  ↓
ejecutar_sql ─────────────── ok ────→ formatear_respuesta → END
  │
  ├── retry (error, intentos < max) → regenerar_sql → ejecutar_sql (loop)
  │
  └── giveup (intentos >= max)      → formatear_fallo     → END
```

Ya está construido en `chat_bd.agent_graph.construir_grafo()`. Lo instanciamos y le pasamos un LLM falso.

In [3]:
from chat_bd.agent_graph import construir_grafo
from chat_bd.llm import obtener_llm
from chat_bd.sql_tools import schema_como_texto

# Escenario FELIZ: el LLM acierta a la primera.
llm_feliz = obtener_llm(
    proveedor="fake",
    respuestas_fake=[
        "SELECT COUNT(*) AS total FROM productos",   # intento 1 - valido
        "Hay 3 productos.",                             # respuesta NL
    ],
)

grafo = construir_grafo(llm_feliz)
estado_final = grafo.invoke({
    "pregunta": "cuantos productos hay?",
    "schema": schema_como_texto(),
    "intentos": 0,
    "max_intentos": 3,
})

print("SQL:     ", estado_final["sql"])
print("Intentos:", estado_final["intentos"])
print("Respuesta:", estado_final["respuesta"])

SQL:      SELECT COUNT(*) AS total FROM productos
Intentos: 1
Respuesta: Hay 3 productos.


## 3 · Escenario de auto-corrección

Ahora el LLM falla a la primera (tabla inventada) y el grafo le pide corregir.

In [4]:
llm_torpe = obtener_llm(
    proveedor="fake",
    respuestas_fake=[
        "SELECT * FROM tabla_que_no_existe",         # intento 1 - falla
        "SELECT COUNT(*) AS total FROM productos",   # intento 2 - corrige
        "Son 3 productos.",                            # respuesta NL
    ],
)

grafo2 = construir_grafo(llm_torpe)
estado2 = grafo2.invoke({
    "pregunta": "cuantos productos hay?",
    "schema": schema_como_texto(),
    "intentos": 0,
    "max_intentos": 3,
})

print("SQL final: ", estado2["sql"])
print("Intentos:  ", estado2["intentos"])
print("Respuesta: ", estado2["respuesta"])

SQL final:  SELECT COUNT(*) AS total FROM productos
Intentos:   2
Respuesta:  Son 3 productos.


## 4 · Escenario "se rinde" (todos los intentos fallan)

Con `max_intentos=3`, si los tres fallan el grafo termina en `formatear_fallo` con un mensaje honesto.

In [5]:
llm_sin_remedio = obtener_llm(
    proveedor="fake",
    respuestas_fake=[
        "SELECT * FROM fake1",
        "SELECT * FROM fake2",
        "SELECT * FROM fake3",
    ],
)

from chat_bd.agent_graph import ejecutar
r = ejecutar("pregunta imposible", llm=llm_sin_remedio, max_intentos=3)
print("exito: ", r.exito)
print("intentos:", r.intentos)
print("error: ", r.error)
print("respuesta:", r.respuesta)

exito:  False
intentos: 3
error:  (sqlite3.OperationalError) no such table: fake3
[SQL: SELECT * FROM fake3 LIMIT 500]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
respuesta: No pude responder tu pregunta despues de 3 intentos. Ultimo error: (sqlite3.OperationalError) no such table: fake3
[SQL: SELECT * FROM fake3 LIMIT 500]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


## Cierre: cuándo usar cada cosa

| Caso | Qué usar |
|---|---|
| Flujo lineal, 1 pasada, el error es aceptable | **LangChain** (chain simple) |
| Flujo lineal con 1–2 tools, sin loops | **LangChain** con tools |
| Estado compartido, loops, routing condicional, varios agentes colaborando | **LangGraph** |

En producción, **la mayoría de agentes útiles necesitan LangGraph** porque los LLMs se equivocan y hay que darles una segunda chance (o una tercera) con feedback estructurado.

**Lo que acabas de ver** es el patrón más común: *generar → ejecutar → (si falla) → regenerar con error → ejecutar → ...*. Variantes del mismo patrón:

- **Self-reflection**: el LLM evalúa su propia respuesta antes de devolverla.
- **Plan-and-execute**: un nodo "planner" rompe la tarea en pasos, un nodo "executor" los hace.
- **Multi-agent**: varios LLMs con roles distintos (crítico, implementador, revisor) coordinados por el grafo.